In [1]:
import h5py
import cv2
import numpy as np
import os
import glob
import sys
import json
from PIL import Image
import matplotlib.pyplot as plt
import logging

sys.path.append("../../../../donut/src/test/")
from common.config import cfg

In [2]:
logging.basicConfig(
    filename="./logs/ImageTransformation.log",
    encoding="utf-8",
    format="%(asctime)s - %(levelname)s - %(message)s",
    level=logging.INFO,
)

In [3]:
# list all image paths
image_dir = "../../../data/image_resize/images/"
image_files = glob.glob(image_dir + "*")
print(len(image_files))
print(image_files[:5])

60575
['../../../data/image_resize/images/45df1fe3293b.jpg', '../../../data/image_resize/images/b2ab3b743d4e.jpg', '../../../data/image_resize/images/51d3b1a6baf3.jpg', '../../../data/image_resize/images/a9e9ce9277c1.jpg', '../../../data/image_resize/images/7f1f545fe081.jpg']


In [4]:
# list all annotation files paths

annotations_dir="../../../data/image_resize/annotations/"
annotations_files=glob.glob(annotations_dir+"*")
print(len(annotations_files))
print(annotations_files[:5])

60575
['../../../data/image_resize/annotations/e91e28111e86.json', '../../../data/image_resize/annotations/75c0449f6917.json', '../../../data/image_resize/annotations/66dd2a250237.json', '../../../data/image_resize/annotations/58595c30beab.json', '../../../data/image_resize/annotations/497a547454d7.json']


In [5]:
# label encoder

label_encoder = {
    "dot": cfg.dot,
    "scatter": cfg.scatter,
    "horizontal_bar": cfg.horizontal_bar,
    "line": cfg.line,
    "vertical_bar": cfg.vertical_bar,
}

print(label_encoder)

{'dot': 0, 'scatter': 1, 'horizontal_bar': 2, 'line': 3, 'vertical_bar': 4}


In [6]:
num_images = len(image_files)

output_hdf5 = "../../../data/classificationH5/ImageTransformation.h5"

with h5py.File(output_hdf5, "w") as hf:
    # support chunking and compressing
    images_ds = hf.create_dataset(
        "images",
        (num_images, cfg.image_width, cfg.image_height, 3),
        dtype=np.uint8,
        chunks=(1, cfg.image_width, cfg.image_height, 3), # suitable for random pick
        compression="gzip",
        compression_opts=4
    )
    
    labels_ds = hf.create_dataset(
        "labels",
        (num_images,),
        dtype=np.int32
    )

    for i, images_pth in enumerate(image_files):
        image_id=images_pth.split('/')[-1].split('.')[0]
        image = cv2.imread(images_pth)
        images_ds[i] = image

        anno_path = [file for file in annotations_files if image_id in file]
        with open(anno_path[0],"r") as f:
            annottation=json.load(f)
            labels_ds[i]=label_encoder[annottation["chart-type"]]
        
        logging.info(f"HDF5 complete: {image_id}")